# Map a storage row into the same domain type

The database calls its columns order_id and amount. The domain calls them id and total. A mapper translates that storage vocabulary at the boundary.

Use the repository environment for this branch. Run cells in order in a fresh kernel; no published-package install is needed.

## Import the types

In [ ]:
!pip install -q aoa-action-machine

In [2]:
from pydantic import Field

from aoa.action_machine.domain import BaseCoreEntity, build
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entities import entities
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources.base_storage import BaseStorage

## Declare the model

In [3]:
class ShopDomain(BaseDomain):
    """Group the shop model declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Customer order", domain=ShopDomain)
class OrderCoreEntity(BaseCoreEntity):
    """Describe an order independently of its storage."""

    id: str = Field(description="Order identifier")
    total: float = Field(ge=0, description="Order total")


@meta(description="Stores the shop entities")
@entities(
    OrderCoreEntity,
)
class ShopStorage(BaseStorage):
    """Hold the shop entities."""

## Run this case

In [4]:
row = {"order_id": "ord-2", "amount": 99.5}
order = build(row, OrderCoreEntity, lambda fields, row: {
    fields.id: row["order_id"],
    fields.total: row["amount"],
})
assert order.id == "ord-2" and order.total == 99.5
print(order.id, order.total)

ord-2 99.5


The action still receives OrderCoreEntity. Selecting a kind does not change hydration: the resource translates field names, while the domain type remains stable.